In [1]:
# Import necessary Spark modules
from pyspark import SparkConf, SparkContext

# Initialize a SparkConf object, which contains configuration information about the Spark application
conf = SparkConf().setAppName("ShakespeareVerbCounter")

# With the configuration created above (conf), initialize a SparkContext object
sc = SparkContext(conf=conf)

In [2]:
# Load files into RDDs
shakespeare_rdd = sc.textFile("shakespeare.txt")
all_verbs_rdd = sc.textFile("all_verbs.txt")
verb_dict_rdd = sc.textFile("verb_dict.txt")

In [3]:
# Collect all verbs from all_verbs_rdd and convert them into a unique set
verb_set = set(all_verbs_rdd.collect())

In [4]:
# Import the string module
import string

# Filter the shakespeare_rdd to retain only non-empty lines
non_empty_lines = shakespeare_rdd.filter(lambda line: line.strip() != "")

# Define a function to clean each line
def clean_line(line):
    # Remove punctuation from the line using a translation table
    line = line.translate(str.maketrans('', '', string.punctuation))
    
    # Remove numeric words from the line
    line = ' '.join([word for word in line.split() if not word.isnumeric()])
    
    # Convert the line to lowercase to standardize the text
    return line.lower()

# Filter out empty lines from shakespeare_rdd and then clean each line using the clean_line function with a map transformation
cleaned_rdd = shakespeare_rdd.filter(lambda line: line.strip() != "").map(clean_line)

In [5]:
# Define a function to filter out verbs from a given line
def filter_verbs(line):
    # Split the line into individual words
    words = line.split()
    
     # Use a list comprehension to filter out words that are present in the verb_set
    return [word for word in words if word in verb_set]

# Apply the filter_verbs function to each line in the cleaned_rdd using the flatMap transformation
verbs_rdd = cleaned_rdd.flatMap(filter_verbs)

In [6]:
def create_mapping(line):
    # Split the input line by commas
    verbs = line.split(',')
    
    # If there are less than two verbs (i.e., no base and derived verb forms), return an empty list
    if len(verbs) < 2: 
        return []
    
    # Assume the first verb in the list is the base verb
    base_verb = verbs[0]
    
    # If the base verb contains non-alphabetical characters, return an empty list
    if not base_verb.isalpha():  
        return []
    
    # For each verb in the list, if it's an alphabetical word, pair it with the base verb
    return [(verb, base_verb) for verb in verbs if verb.isalpha()]

# Use the flatMap transformation to apply the create_mapping function to each line in the RDD
mappings_rdd = verb_dict_rdd.flatMap(create_mapping)

# Convert the mappings RDD to a dictionary and collect it to the driver node
verb_dictionary = dict(mappings_rdd.collect())

# Broadcast the verb dictionary to all the executor nodes
broadcast_verb_dict = sc.broadcast(verb_dictionary)

In [7]:
# Define a function to get the base form of a verb
def to_base_verb(word):
    return broadcast_verb_dict.value.get(word, word)                       

# Define a function to check if a word is valid
def is_valid_word(word):
    return word.isalpha() and word in broadcast_verb_dict.value

# Split lines in cleaned_rdd into words, filter using is_valid_word, convert valid words to their base form, and store the result in base_verb_rdd
base_verb_rdd = cleaned_rdd.flatMap(lambda line: line.split()).filter(is_valid_word).map(to_base_verb)

In [8]:
# Transform the base_verb_rdd into a pair RDD where each verb is paired with the number 1
verb_counts = base_verb_rdd.map(lambda verb: (verb, 1)).reduceByKey(lambda a, b: a + b)

In [9]:
# Sort 'verb_counts' by count in descending order and retrieve the top 10 verbs
top_10_words = verb_counts.sortBy(lambda pair: -pair[1]).take(10)

# Print each of the top 10 verbs along with their counts
for word, count in top_10_words:
    print(f"{word}: {count}")

be: 26727
have: 7848
do: 6416
come: 3610
make: 2892
love: 2501
go: 2476
let: 2384
say: 2356
know: 2251
